In [1]:
import pandas as pd
from pathlib import Path

In [2]:
df = pd.read_csv("TableS6_AugmentedDataset.csv")
df_source = pd.read_csv("TableS6_AugmentedDataset_source.csv")
#df_source.rename(colnames={"assigned_subcat_new": "assigned_subcat"}, inplace=True)
df_new = pd.read_csv("../../new_data_phageGAP/tables/TableS6_AugmentedDataset.csv")

In [3]:
print(df_source.columns)

Index(['phage_ID', 'organism', 'protein_ID', 'product', 'note', 'annot',
       'vfdb_short_name', 'CARD_short_name', 'AD_genes_type',
       'assigned_subcat', 'assigned_top', 'table_origin', 'product_cleaned',
       'note_cleaned', 'product_source', 'assigned_subcat_new'],
      dtype='str')


In [4]:
(df_source["assigned_subcat"] == df_source["assigned_subcat_new"]).all()

np.True_

In [5]:
mask = df_source["assigned_subcat"] != df_source["assigned_subcat_new"]

print(f"Different: {mask.sum()}")
print(f"Same: {(~mask).sum()}")

Different: 0
Same: 94227


In [6]:
df_source.loc[mask, ["table_origin"]].value_counts()

Series([], Name: count, dtype: int64)

In [7]:
df_source.loc[mask]

,phage_ID,organism,protein_ID,product,note,annot,vfdb_short_name,CARD_short_name,AD_genes_type,assigned_subcat,assigned_top,table_origin,product_cleaned,note_cleaned,product_source,assigned_subcat_new


In [8]:
df.loc[mask, ["assigned_subcat"]].join(
    df_source.loc[mask, ["assigned_subcat"]],
    lsuffix="_df",
    rsuffix="_source"
)

,assigned_subcat_df,assigned_subcat_source


In [9]:
print(df_source['product_source'].unique())

<StringArray>
[             'annot (Pharokka)',                       'product',
 'AD_genes_type (DefenseFinder)',                          'note',
        'vfdb_short_name (VFDB)',        'CARD_short_name (CARD)']
Length: 6, dtype: str


In [10]:
description_cols_map = {
    'annot (Pharokka)': "annot",
    'vfdb_short_name (VFDB)': "vfdb_short_name",
    'CARD_short_name (CARD)': "CARD_short_name",
    'AD_genes_type (DefenseFinder)': "AD_genes_type",
    'product': "product",
    'note': 'note'
}

annotation_map = {
    'annot (Pharokka)': "Pharokka PHROG",
    'vfdb_short_name (VFDB)': "Pharokka VFDB",
    'CARD_short_name (CARD)': "Pharokka CARD",
    'AD_genes_type (DefenseFinder)': "DefenseFinder",
    'product': "Genbank product",
    'note': 'Genbank note'
}

In [11]:
df_source["source"] = df_source["product_source"].map(annotation_map)
df_source["annotation"] = df_source.apply(lambda row: row[description_cols_map.get(row["product_source"])], axis=1)

In [14]:
df_source[["protein_ID", "source", "annotation"]].to_csv("AugmentedDataset_source_annotated.csv", index=False)

In [ ]:
dfs_by_category = {category: group.copy() for category, group in df_source.groupby("assigned_subcat")}

In [ ]:
output_dir = Path("unique_descriptions")
output_dir.mkdir(exist_ok=True)

for category, df_category in dfs_by_category.items():

    df_category = df_category.copy()

    # prioritized description for each row
    def get_description(row):
        return row[description_cols_map.get(row["product_source"], "product")], row["product_source"]

    # apply prioritization
    df_category[["description", "source"]] = df_category.apply(get_description, axis=1, result_type="expand")

    # remove rows without a description
    descriptions = df_category.dropna(subset=["description"])

    # count descriptions by table_origin
    counts = (pd.crosstab([descriptions["description"], descriptions["source"]], descriptions["table_origin"]).reset_index())

    # make sure all expected table_origin columns exist
    for origin in ["phg", "bac", "vir"]:
        if origin not in counts.columns:
            counts[origin] = 0

    # total count
    counts["count"] = counts[["phg", "bac", "vir"]].sum(axis=1)

    # reorder columns
    counts = counts[["description", "source", "count", "phg", "bac", "vir"]]

    # order by total count
    counts = counts.sort_values("count", ascending=False)

    # save
    filename = output_dir / f"{category}.csv"
    counts.to_csv(filename, index=False)